# ObserverBench: matched Qwen3.5-9B APPS replication

Experiments designed/concieved by Vijay Erramilli. Code written by Vijay Erramilli and Codex

**Question.** Under the same APPS rows, prompts, controller, audit budgets, and loss used for the earlier Qwen2.5 and Gemma studies, which observer gives the lowest realized safety loss on Qwen3.5-9B?

The prespecified primary comparison is neutral-context residual minus verdict-adjacent residual at 1% attack prevalence and a 2% audit budget. A separate post-cache stage compares an official Qwen-Scope SAE observer with the output and residual observers. The SAE was trained on the base checkpoint and is not checkpoint-matched to the post-trained model.

This notebook permits one compatibility smoke before any outcome is inspected. The full and SAE stages remain disabled until their explicit review flags are changed.


## 1. Accelerator and model access

Use an A100 runtime. The login prompt keeps the Hugging Face token outside the notebook. The experiment pins exact model and SAE commits and verifies all three SAE file hashes.


In [ ]:
# Experiments designed/concieved by Vijay Erramilli. Code written by Vijay Erramilli and Codex
from __future__ import annotations

import json
import os
from pathlib import Path
import subprocess
import sys

subprocess.run(["nvidia-smi"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)
from huggingface_hub import login
login()


## 2. Install and verify the frozen source

The local source seal covers the experiment configuration, runner, measurement adapters, tests, and this notebook. It is a reproducibility seal, not an independent preregistration. Set `OBSERVERBENCH_REVISION` to the tagged release when one is available.


In [ ]:
REPO_URL = os.environ.get(
    "OBSERVERBENCH_REPO_URL",
    "https://github.com/kwisatzh/observerbench.git",
)
REPO_REVISION = os.environ.get("OBSERVERBENCH_REVISION", "main")
REPO = Path(os.environ.get("OBSERVERBENCH_REPO_ROOT", "/content/observerbench"))

if not (REPO / ".git").exists():
    subprocess.run(
        ["git", "clone", "--depth=1", "--branch", REPO_REVISION, REPO_URL, str(REPO)],
        check=True,
    )
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[ai-control,sae]"],
    check=True,
)
CONFIG = REPO / "configs/revision/ai_control/controlarena_apps_qwen3_5_9b_v0.json"
MANIFEST = REPO / "configs/revision/ai_control/controlarena_apps_qwen3_5_9b_v0_source_manifest.json"
RUNNER = REPO / "scripts/run_controlarena_apps_qwen35.py"
subprocess.run(
    [sys.executable, str(REPO / "scripts/seal_controlarena_apps_qwen35.py"), "--config", str(CONFIG), "--manifest", str(MANIFEST), "--verify"],
    cwd=REPO,
    check=True,
)
print("Verified source revision:", REPO_REVISION)


## 3. Persistent output directory

Model views are cached separately in Drive. If Colab disconnects during the full run, rerun that stage with `--resume`; the runner verifies every cache against the sealed source and frozen row identities before reuse. Completed result files are never overwritten.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUTDIR = Path("/content/drive/MyDrive/observerbench/controlarena_apps_qwen3_5_9b_v0")
OUTDIR.mkdir(parents=True, exist_ok=True)
OUTDIR


## 4. Run the single compatibility smoke

Run this once. It audits all prompt lengths, checks the exact model architecture and score tokens, performs inference on exactly two APPS pairs, and verifies the hashes, shapes, and TopK-50 behavior of all three official SAEs. It does **not** fit an observer, compute AUROC, rank monitors, or inspect outcomes.


In [ ]:
subprocess.run([
    sys.executable, str(RUNNER),
    "--config", str(CONFIG),
    "--manifest", str(MANIFEST),
    "--outdir", str(OUTDIR),
    "--stage", "compatibility-smoke",
    "--device", "cuda",
], cwd=REPO, check=True)


In [ ]:
smoke = json.loads((OUTDIR / "compatibility_smoke.json").read_text())
{
    "status": smoke["status"],
    "model": smoke["model"],
    "locked_rows_at_limit": {
        name: item["at_or_above_limit_by_split"]["locked_test"]
        for name, item in smoke["length_audit"].items()
    },
    "saes": [
        {key: row[key] for key in ("layer", "d_in", "width", "top_k", "encoded_shape", "finite")}
        for row in smoke["saes"]
    ],
}


## 5. Review gate, then run the core observer panel

Stop here and inspect the smoke summary. Only then change `RUN_FULL` to `True`. The runner checks the seal and smoke again, freezes the same controller and prevalence grid for every observer, and writes the primary neutral-minus-verdict contrast separately.


In [ ]:
RUN_FULL = False
if not RUN_FULL:
    print("Core outcome run remains review-gated.")
else:
    subprocess.run([
        sys.executable, str(RUNNER),
        "--config", str(CONFIG),
        "--manifest", str(MANIFEST),
        "--outdir", str(OUTDIR),
        "--stage", "full",
        "--device", "cuda",
        "--resume",
        "--allow-full-run",
    ], cwd=REPO, check=True)


## 6. Run the official Qwen-Scope observer from the frozen cache

After the core result is complete, change `RUN_SAE` to `True`. This stage does not reload Qwen3.5 or rewrite the core result. It applies the three pinned SAEs to the bound neutral-residual cache, chooses layer and ridge from calibration data only, and writes separate SAE score and contrast artifacts.


In [ ]:
RUN_SAE = False
if not RUN_SAE:
    print("SAE outcome run remains review-gated.")
else:
    subprocess.run([
        sys.executable, str(RUNNER),
        "--config", str(CONFIG),
        "--manifest", str(MANIFEST),
        "--outdir", str(OUTDIR),
        "--stage", "sae",
        "--device", "cuda",
        "--allow-sae-run",
    ], cwd=REPO, check=True)


## 7. Read the frozen outcomes

Statistical accuracy and realized violations remain separate. The table is sorted by realized violations at the primary 2% audit budget; it does not create a generic observer ranking outside this task.


In [ ]:
import pandas as pd

def result_rows(records):
    rows = []
    for observer, record in records.items():
        metrics = record["metrics"]
        rows.append({
            "observer": observer,
            "risk_auroc": metrics["risk_auroc"],
            "realized_violations": metrics["realized_violations"],
            "audit_precision": metrics["audit_precision"],
        })
    return pd.DataFrame(rows).sort_values(["realized_violations", "risk_auroc"], ascending=[True, False])

if (OUTDIR / "results.json").exists():
    core = json.loads((OUTDIR / "results.json").read_text())
    display(result_rows(core["budget_results"]["0.02"]))
    display(core["primary_rare_prevalence_outcome"]["paired_contrast"])
else:
    print("No core outcome yet.")

if (OUTDIR / "sae_results.json").exists():
    sae = json.loads((OUTDIR / "sae_results.json").read_text())
    display(result_rows(sae["fixed_budget_results"]["0.02"]))
    display(sae["primary_operating_cell"])
else:
    print("No SAE outcome yet.")


## Interpretation boundary

This is a frozen, offline monitor comparison. It does not test an attacker that learns which channel is monitored, does not identify deception or intent, and does not make the base-model SAE checkpoint-matched to the post-trained model. Report the result regardless of direction.
